# Baseline Model: Dummy Classifier

To set up a **Dummy Classifier** as your baseline model, we want to establish a worst-case or trivial performance benchmark. For a highly imbalanced dataset like the Kaggle Credit Card Fraud dataset (0.17% fraud), this is incredibly critical.

If a model simply predicts **"Not Fraud"** (`0`) for every single transaction, it will achieve **99.83% accuracy**, which sounds amazing but is completely useless because it catches **0% of actual fraud**. A Dummy Classifier helps formally document this behavior so you can prove your future machine learning models (like Random Forests, XGBoost, or Neural Networks) are actually learning something useful.

In [5]:
# Load the credit card dataset and define the feature columns
import pandas as pd
import os
data_path = "../data/raw/creditcard.csv"

if not os.path.exists(data_path):
    print("Raw data missing. Attempting to download via src/dataloader.py...")
    # This runs the script from the context of your notebook directory
    # Adjust the relative path to the script if necessary (e.g., ../src/dataloader.py)

    output_dir = os.path.dirname(data_path)

    os.system(f"python ../src/data_loader.py --download --output {output_dir}")


credit_card_data = pd.read_csv('../data/raw/creditcard.csv')

Raw data missing. Attempting to download via src/dataloader.py...


In [7]:
import sys
import os
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report

# Make sure the src package is importable from the notebook directory
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
from evaluation import evaluate_model_performance

X = credit_card_data.drop(columns=["Class"])
y = credit_card_data["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

dummy = DummyClassifier(strategy="stratified", random_state=42)
dummy.fit(X_train, y_train)

y_pred = dummy.predict(X_test)
y_pred_proba = dummy.predict_proba(X_test)[:, 1]

metrics = evaluate_model_performance(y_test, y_pred, y_pred_proba)

print("Class distribution in test set:")
print(y_test.value_counts(normalize=True))
print()

print("Dummy Classifier metrics from src/evaluation.py:")
for name, value in metrics.items():
    print(f"{name.replace('_', ' ').title():<10}: {value:.4f}")
print()
print("Classification report:")
print(classification_report(y_test, y_pred, zero_division=0))

Class distribution in test set:
Class
0    0.99828
1    0.00172
Name: proportion, dtype: float64

Dummy Classifier metrics from src/evaluation.py:
Accuracy  : 0.9966
Precision : 0.0000
Recall    : 0.0000
F1 Score  : 0.0000
Roc Auc   : 0.4992
Pr Auc    : 0.0017

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.00      0.00      0.00        98

    accuracy                           1.00     56962
   macro avg       0.50      0.50      0.50     56962
weighted avg       1.00      1.00      1.00     56962



## Dummy Classifier Performance Report

### Executive Summary
The Dummy Classifier with a "stratified" strategy serves as our baseline model. While it achieves a deceptively high accuracy of **99.66%**, it completely fails to detect any fraudulent transactions, making it a poor predictor for fraud detection.

### Key Findings

#### Class Imbalance
The test set exhibits severe class imbalance:
- **Non-fraud (Class 0):** 99.83% of transactions (56,864 cases)
- **Fraud (Class 1):** 0.17% of transactions (98 cases)

This extreme imbalance explains why a trivial "always predict non-fraud" strategy achieves such high accuracy.

#### Model Performance Metrics

| Metric | Value | Interpretation |
|--------|-------|-----------------|
| **Accuracy** | 99.66% | Misleading due to class imbalance |
| **Precision** | 0.00% | No fraudulent transactions identified |
| **Recall** | 0.00% | Catches 0% of actual fraud cases |
| **F1-Score** | 0.00% | Poor balance between precision and recall |
| **ROC-AUC** | 0.50 | Random guessing performance |
| **PR-AUC** | 0.0017 | Near-zero ability to separate classes |

#### Classification Report Analysis
- The model predicts **0 (non-fraud) for all 56,962 test samples**
- It correctly classifies all 56,864 legitimate transactions
- It **misses all 98 fraudulent transactions** (0% recall on fraud)
- The high weighted average metrics are artifacts of class imbalance

### Conclusion
This baseline demonstrates the critical need for specialized fraud detection models. Any machine learning model must achieve **better than 0% recall on fraud cases** to provide business value. The subsequent models (Random Forest, XGBoost, Neural Networks) should be evaluated on metrics like **Recall, Precision, F1-Score, and ROC-AUC** rather than raw accuracy.